# 🏛️ QUỸ ĐẦU TƯ CHỨNG KHOÁN AI - UNIVERSAL QUANT ALPHA MODEL
### Pipeline Huấn luyện Mô hình Machine Learning (LightGBM) trên Kaggle / Google Colab

Notebook này được thiết kế để huấn luyện **Bộ não Dự báo Thị trường Phổ quát (Universal Quant Alpha Model)**:
1. **Dữ liệu đầu vào**: `ml_dataset_train.parquet` (hoặc `.csv`) gồm 15 đặc trưng kỹ thuật & dòng tiền chuẩn hóa (Stationary Features).
2. **Phương pháp gán nhãn**: **Triple-Barrier Method** (Rào cản chốt lời +5%, cắt lỗ -3% trong 5 phiên T+2.5).
3. **Phân chia dữ liệu**: **Purged Time-Series Split** (Tuyệt đối không rò rỉ tương lai - Zero Look-Ahead Bias).
4. **Thuật toán**: **LightGBM Classifier** (Gradient Boosted Trees siêu nhanh & tối ưu cho dữ liệu dạng bảng).
5. **Đầu ra**: File `universal_alpha_model.joblib` tải về thả trực tiếp vào thư mục `models/` của Quỹ.

In [ ]:
# 1. KHỞI TẠO MÔI TRƯỜNG & CÀI ĐẶT THƯ VIỆN
import os
import sys
import json
from datetime import datetime
import numpy as np
import polars as pl
import lightgbm as lgb
import joblib
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
import matplotlib.pyplot as plt

print(f"✅ Đã nạp thư viện thành công! LightGBM version: {lgb.__version__}")

In [ ]:
# 2. NẠP DỮ LIỆU ĐÃ TRÍCH XUẤT TỪ LOCAL
# Nếu chạy trên Kaggle, tải file ml_dataset_train.parquet (hoặc csv) lên mục Input Data
DATA_PATH = "ml_dataset_train.parquet"
if not os.path.exists(DATA_PATH):
    if os.path.exists("ml_dataset_train.csv"):
        DATA_PATH = "ml_dataset_train.csv"
    else:
        # Đường dẫn giả định trên Kaggle /kaggle/input/...
        DATA_PATH = "../input/ai-investment-fund/ml_dataset_train.parquet"

if DATA_PATH.endswith(".parquet"):
    df = pl.read_parquet(DATA_PATH)
else:
    df = pl.read_csv(DATA_PATH)

print(f"📊 Tổng số mẫu dữ liệu nạp vào: {len(df):,} dòng")
print(f"🏢 Số lượng mã cổ phiếu: {df['ticker'].n_unique()} mã ({df['ticker'].unique().to_list()})")
print(f"⏱️ Chuỗi thời gian: từ {df['trade_date'].min()} đến {df['trade_date'].max()}")

In [ ]:
# 3. ĐỊNH NGHĨA DANH SÁCH 15 ĐẶC TRƯNG CHUẨN HÓA (FEATURES)
FEATURES = [
    "ret_1d", "ret_3d", "ret_5d", "ret_10d", "ret_20d",
    "rel_ret_5d", "rel_ret_20d",
    "vol_ratio_20", "vol_zscore_60",
    "norm_atr_14", "bb_width_20",
    "body_to_range", "upper_wick_ratio", "lower_wick_ratio",
    "rsi_14", "macd_hist_norm"
]

TARGET = "target"

# Kiểm tra tỷ lệ phân phối nhãn (Class Balance)
pos_count = df.filter(pl.col(TARGET) == 1).shape[0]
neg_count = df.filter(pl.col(TARGET) == 0).shape[0]
print(f"🎯 Nhãn 1 (Thương vụ Thắng +5%): {pos_count:,} ({pos_count/len(df)*100:.2f}%)")
print(f"🛡️ Nhãn 0 (Thương vụ Thua / Hòa): {neg_count:,} ({neg_count/len(df)*100:.2f}%)")

In [ ]:
# 4. PHÂN CHIA DỮ LIỆU CHUỖI THỜI GIAN (PURGED TIME-SERIES SPLIT)
# Tuyệt đối không dùng random train_test_split vì sẽ rò rỉ thông tin tương lai (Look-ahead bias)
# Ta chia 80% thời gian đầu làm Train, 20% thời gian sau làm Test (Out-of-Sample)

unique_dates = df["trade_date"].unique().sort().to_list()
split_idx = int(len(unique_dates) * 0.80)
split_date = unique_dates[split_idx]

print(f"📅 Điểm cắt thời gian (Split Date): {split_date}")
train_df = df.filter(pl.col("trade_date") < split_date)
test_df = df.filter(pl.col("trade_date") >= split_date)

print(f"🏋️ Tập Huấn luyện (Train Set): {len(train_df):,} mẫu ({train_df['trade_date'].min()} -> {train_df['trade_date'].max()})")
print(f"🧪 Tập Kiểm thử (Test Set):    {len(test_df):,} mẫu ({test_df['trade_date'].min()} -> {test_df['trade_date'].max()})")

X_train = train_df.select(FEATURES).to_pandas()
y_train = train_df[TARGET].to_numpy()

X_test = test_df.select(FEATURES).to_pandas()
y_test = test_df[TARGET].to_numpy()

In [ ]:
# 5. HUẤN LUYỆN MÔ HÌNH LIGHTGBM CHỐNG OVERFITTING
# Sử dụng các tham số Regularization mạnh mẽ để chống học vẹt dữ liệu tài chính

scale_pos_weight = float(neg_count / max(pos_count, 1))

model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=5,
    num_leaves=25,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,         # L1 Regularization
    reg_lambda=2.0,        # L2 Regularization
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

print("⏳ Đang huấn luyện LightGBM trên tập Train...")
model.fit(
    X_train,
    y_train,
    eval_X=X_test,
    eval_y=y_test,
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
)
print(f"✅ Huấn luyện hoàn tất! Số cây tốt nhất (Best Iteration): {model.best_iteration_}")

In [ ]:
# 6. ĐÁNH GIÁ CHẤT LƯỢNG MÔ HÌNH TRÊN TẬP OUT-OF-SAMPLE (TEST SET)
y_pred_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)

print("=" * 60)
print(f"🏆 ROC-AUC SCORE TRÊN TẬP TEST (OUT-OF-SAMPLE): {auc:.4f}")
print("=" * 60)

# Đánh giá Precision@Top-K (Tỷ lệ thắng nếu chỉ chọn các cổ phiếu có xác suất cao nhất)
test_eval_df = test_df.select(["ticker", "trade_date", TARGET, "forward_ret_5d"]).to_pandas()
test_eval_df["ml_score"] = y_pred_proba

for threshold in [0.50, 0.60, 0.65, 0.70]:
    subset = test_eval_df[test_eval_df["ml_score"] >= threshold]
    if len(subset) > 0:
        win_rate = (subset[TARGET].sum() / len(subset)) * 100.0
        avg_ret = subset["forward_ret_5d"].mean() * 100.0
        print(f"⭐ Ngưỡng ML_Score >= {threshold:.2f} (Số lệnh: {len(subset):4d}) | Tỷ lệ Thắng: {win_rate:5.1f}% | Lãi TB 5 phiên: {avg_ret:+5.2f}%")
    else:
        print(f"⭐ Ngưỡng ML_Score >= {threshold:.2f} | Không có lệnh nào thỏa mãn")

In [ ]:
# 7. TRỰC QUAN HÓA FEATURE IMPORTANCE (ĐẶC TRƯNG NÀO TẠO RA ALPHA?)
importance = model.feature_importances_
indices = np.argsort(importance)

plt.figure(figsize=(10, 6))
plt.title("ĐỘ QUAN TRỌNG CỦA CÁC ĐẶC TRƯNG (FEATURE IMPORTANCE)", fontsize=14, fontweight="bold")
plt.barh(range(len(indices)), importance[indices], color="#1f77b4", align="center")
plt.yticks(range(len(indices)), [FEATURES[i] for i in indices])
plt.xlabel("Tần suất phân tách (Split Importance)")
plt.tight_layout()
plt.show()

In [ ]:
# 8. ĐÓNG GÓI VÀ XUẤT FILE MODEL ĐỂ SỬ DỤNG TRÊN LOCAL
OUTPUT_MODEL_FILE = "universal_alpha_model.joblib"

package = {
    "model": model,
    "features": FEATURES,
    "metrics": {
        "test_auc": float(auc),
        "best_iteration": int(model.best_iteration_),
        "train_samples": int(len(X_train)),
        "test_samples": int(len(X_test))
    },
    "trained_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "algorithm": "LightGBM Classifier"
}

joblib.dump(package, OUTPUT_MODEL_FILE, compress=3)
file_size_kb = os.path.getsize(OUTPUT_MODEL_FILE) / 1024

print("=" * 60)
print(f"🎉 ĐÃ XUẤT FILE MODEL THÀNH CÔNG: {OUTPUT_MODEL_FILE}")
print(f"📦 Dung lượng tệp: {file_size_kb:.1f} KB (Siêu nhẹ, tải về và thả vào thư mục models/ của dự án)")
print("=" * 60)